# Entrenamiento y evaluación de modelos

**Objetivo:** entrenar modelos que, con la información disponible hasta cierto mes del semestre, identifiquen a los estudiantes en riesgo de reprobar Pensamiento Cuantitativo.

| | |
|---|---|
| **Entradas** | `Datos/entrenamiento.csv`, `Datos/validacion.csv` y `Datos/prueba.csv` (generados en [`analisis_exploratorio.ipynb`](../exploratorio/analisis_exploratorio.ipynb)) |
| **Salidas** | `resultados_clasificacion/` y `resultados_regresion/`: métricas, importancias, gráficas y modelos (`.pkl`) por mes y algoritmo |
| **Funciones auxiliares** | [`utils.py`](utils.py) |
| **Siguiente paso** | [`graficas_resultados.ipynb`](graficas_resultados.ipynb) |

### Planteamiento
- **Un modelo por mes (enero a abril):** el modelo del mes *k* solo usa la información disponible hasta ese mes, simulando una alerta temprana. Mayo no se modela: para entonces ya se conoce casi toda la nota y no queda tiempo para intervenir.
- **Tarea principal: clasificación** de `Aprobado`, con *reprobado* (0) como clase de interés. Es la que usa la aplicación.
- **Tarea complementaria: regresión** de `NotaFinal`.
- **Cinco algoritmos por tarea:** árbol de decisión, Random Forest, LightGBM, XGBoost y regresión logística (regresión lineal en la tarea de regresión), desde modelos interpretables hasta *ensembles* de *boosting*.

### Decisiones de diseño
| Decisión | Justificación |
|---|---|
| Optimizar la **PR-AUC de la clase reprobado** en validación | Con ~14 % de reprobados, la exactitud y el AUC-ROC son engañosamente altos; la curva precisión-recall se enfoca en la clase minoritaria |
| Elegir el **umbral de decisión que maximiza F2** en validación | F2 pondera el *recall* el doble que la precisión: en una alerta temprana es más costoso no detectar a un estudiante en riesgo que dar una falsa alarma |
| En clasificación, manejar el desbalance con el **umbral** y no con pesos ni sobremuestreo | No distorsiona las probabilidades del modelo y concentra la decisión de sensibilidad en un solo parámetro |
| **Optuna** (muestreador TPE, 1,000 *trials*, semilla 42) | Búsqueda bayesiana, más eficiente que una búsqueda en malla en espacios de hiperparámetros grandes |
| **Target encoding** de `Colegio` y `TituloNivelMedio`, ajustado solo con entrenamiento | Tienen muchas categorías y *one-hot* crearía decenas de columnas dispersas; ajustarlo solo con entrenamiento evita la fuga de información |
| Escalado (`StandardScaler`) solo en los modelos lineales | Los modelos basados en árboles no son sensibles a la escala |
| Evaluación final en el **conjunto de prueba**, que no se usa en ningún otro paso | Estimación honesta del desempeño |

In [1]:
# Librerías
import os
import pickle
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn import svm
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc as auc_score,
    mean_squared_error,
    mean_absolute_error,
    make_scorer,
    average_precision_score,
    fbeta_score
)
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.linear_model import SGDClassifier, SGDRegressor

from utils import *
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Datos y variables
Cargamos los tres conjuntos y definimos las variables de cada modelo mensual:

| Modelo | Variables |
|---|---|
| Mes 1 (enero) | `PAA`, `Colegio`, `TituloNivelMedio`, `veces_curso`, `porcentaje_acum_actividades_mes_1` |
| Mes 2 (febrero) | Las mismas (con el % acumulado al mes 2) + `nota_examen_mes_2` (Parcial 1) |
| Mes 3 (marzo) | … + `nota_examen_mes_3` (Parcial 2) |
| Mes 4 (abril) | … + `nota_examen_mes_4` (Parcial 3) |

**Variables excluidas** (con base en [`analisis_exploratorio.ipynb`](../exploratorio/analisis_exploratorio.ipynb)):
- Puntuaciones z: redundantes con su versión original (VIF alto).
- Entregas tardías, cantidad de comentarios y deltas de actividades: sin diferencias significativas consistentes entre aprobados y reprobados.
- Velocidad de calificación (`DiferenciaPromedio`): significancia marginal (p = 0.049).
- `Genero`: asociación muy débil con el resultado (V de Cramér = 0.06); además, usarlo para clasificar a un estudiante plantea problemas de equidad.
- `Virtualidad`: refleja una condición excepcional (la pandemia) que no se espera en los semestres en que se usaría la herramienta.

`preprocess_features` aplica el *target encoding* (y el escalado, si corresponde) ajustándolo solo con el conjunto de entrenamiento.

In [2]:
# Carga de los conjuntos y variables de cada modelo mensual
datos = pd.read_csv('../../Datos/entrenamiento.csv')
train = pd.read_csv('../../Datos/entrenamiento.csv')
val = pd.read_csv('../../Datos/validacion.csv')
test =pd.read_csv('../../Datos/prueba.csv')


# Variables objetivo
y_clasificacion_train = train['Aprobado']
y_clasificacion_val = val['Aprobado']
y_clasificacion_test = test['Aprobado']

y_regresion_train = train['NotaFinal']
y_regresion_val = val['NotaFinal']
y_regresion_test = test['NotaFinal']
COLS_TARGET_ENCODING = ["TituloNivelMedio", "Colegio"]

POS_LABEL = 1
NEG_LABEL = 0

N_TRIALS = 1000
 

def preprocess_features(X_train, X_test, y_train, target_type="auto", escalar=False):
    """Aplica target encoding a Colegio y TituloNivelMedio y, si se indica, escala las variables"""
    X_train = X_train.copy().reset_index(drop=True)
    X_test = X_test.copy().reset_index(drop=True)

    te = TargetEncoder(target_type=target_type, random_state=42)
    X_train[COLS_TARGET_ENCODING] = te.fit_transform(X_train[COLS_TARGET_ENCODING], y_train)
    X_test[COLS_TARGET_ENCODING] = te.transform(X_test[COLS_TARGET_ENCODING])

    scaler = None
    if escalar:
        scaler = StandardScaler()
        cols = X_train.columns
        X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=cols)
        X_test = pd.DataFrame(scaler.transform(X_test), columns=cols)

    return X_train, X_test, te, scaler

# Variables de cada modelo mensual
variables = {}

base_variables = ['PAA', 'Colegio', 'TituloNivelMedio',  'veces_curso']


for i in range(1, 5):
    cols = base_variables.copy()
    cols.append(f'porcentaje_acum_actividades_mes_{i}')
    # cols.append(f'z_porcentaje_acum_actividades_mes_{i}')
    cols += [f'nota_examen_mes_{j}' for j in range(2, i + 1)]   # parciales disponibles hasta el mes i
    variables[f"mes{i}"] = cols



In [3]:
# Títulos de nivel medio en entrenamiento
datos['TituloNivelMedio'].value_counts()

TituloNivelMedio
BACHILLER EN CIENCIAS Y LETRAS                                         1063
BACHILLER EN CIENCIAS Y LETRAS CON ORIENTACION EN COMPUTACION           258
OTRO                                                                     85
BACHILLER EN COMPUTACION                                                 50
BACHILLER EN COMPUTACION CON ORIENTACION CIENTIFICA                      25
BACHILLER BILINGUE EN CIENCIAS                                           21
DIPLOMA NIVEL MEDIO 3RO BASICO                                           17
HIGH SCHOOL                                                              16
BACHILLERATO EN CIENCIAS Y LETRAS EN COMPUTACION                         16
BACHILLERATO EN CIENCIAS Y LETRAS CON ORIENTACION EN DISENO GRAFICO      14
BACHILLER INTERNACIONAL                                                  13
PERITO EN COMPUTACION                                                    12
PERITO EN ELECTRONICA DIGITAL                                          

Definimos los espacios de búsqueda de hiperparámetros de cada algoritmo para Optuna. Los rangos son amplios a propósito, para que la búsqueda explore tanto modelos simples como complejos.

In [4]:
# Espacios de búsqueda de hiperparámetros
def space_dt_clf(trial):
    """Espacio de búsqueda del árbol de decisión para clasificación"""
    return {
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
        "max_depth": trial.suggest_int("max_depth", 2, 32),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 60),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 60),
        "max_features": trial.suggest_categorical("max_features", [None, "sqrt", "log2"]),
        "ccp_alpha": trial.suggest_float("ccp_alpha", 0.0, 0.05),
    }
 
 
def space_dt_reg(trial):
    """Espacio de búsqueda del árbol de decisión para regresión"""
    return {
        "criterion": trial.suggest_categorical("criterion", ["squared_error", "friedman_mse", "absolute_error"]),
        "max_depth": trial.suggest_int("max_depth", 2, 32),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 60),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 60),
        "max_features": trial.suggest_categorical("max_features", [None, "sqrt", "log2"]),
        "ccp_alpha": trial.suggest_float("ccp_alpha", 0.0, 0.05),
    }
 
 
def space_rf_clf(trial):
    """Espacio de búsqueda de Random Forest para clasificación"""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 50, 800),
        "max_depth": trial.suggest_int("max_depth", 2, 32),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 60),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 60),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
    }
 
 
def space_rf_reg(trial):
    """Espacio de búsqueda de Random Forest para regresión"""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "max_depth": trial.suggest_int("max_depth", 2, 32),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "criterion": trial.suggest_categorical("criterion", ["squared_error", "friedman_mse", "absolute_error"]),
    }
 
 
def space_lgbm(trial):
    """Espacio de búsqueda de LightGBM"""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 7, 64),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }


def space_xgb(trial):
    """Espacio de búsqueda de XGBoost"""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
 
 
def space_logreg(trial):
    """Espacio de búsqueda de la regresión logística"""
    return {
        "C": trial.suggest_float("C", 1e-4, 1e2, log=True),
        "penalty": trial.suggest_categorical("penalty", ["l1", "l2"]),
    }

Definimos la configuración de cada algoritmo de clasificación: clase del modelo, espacio de búsqueda y parámetros fijos. Usamos `random_state = 42` en todos para que los resultados sean reproducibles; la regresión logística usa el *solver* `liblinear` porque admite penalizaciones L1 y L2. En XGBoost se dejó desactivado `scale_pos_weight`, ya que el desbalance se maneja con el umbral de decisión.

In [ ]:
# Configuración de los modelos de clasificación
CLASSIFICATION_MODELS = {
    "decision_tree": {
        "model_class": DecisionTreeClassifier,
        "param_space": space_dt_clf,
        "fixed_kwargs": {"random_state": 42},
    },
    "random_forest": {
        "model_class": RandomForestClassifier,
        "param_space": space_rf_clf,
        "fixed_kwargs": {"random_state": 42, "n_jobs": -1},
    },
    "lightgbm": {
        "model_class": lgb.LGBMClassifier,
        "param_space": space_lgbm,
        "fixed_kwargs": {"random_state": 42, "verbosity": -1, "n_jobs": -1},
    },
    "xgboost": {
        "model_class": xgb.XGBClassifier,
        "param_space": space_xgb,
        "fixed_kwargs": lambda y_train: {
            "random_state": 42,
            "eval_metric": "logloss",
            "n_jobs": -1,
        },
    },
    "logistic_regression": {
        "model_class": LogisticRegression,
        "param_space": space_logreg,
        "fixed_kwargs": {"random_state": 42, "max_iter": 2000, "solver": "liblinear"},
        "escalar": True
    },
}

Configuración de los algoritmos de regresión. La regresión lineal no tiene hiperparámetros que optimizar.

In [6]:
# Configuración de los modelos de regresión
REGRESSION_MODELS = {
    "decision_tree": {
        "model_class": DecisionTreeRegressor,
        "param_space": space_dt_reg,
        "fixed_kwargs": {"random_state": 42},
    },
    "random_forest": {
        "model_class": RandomForestRegressor,
        "param_space": space_rf_reg,
        "fixed_kwargs": {"random_state": 42, "n_jobs": -1},
    },
    "lightgbm": {
        "model_class": lgb.LGBMRegressor,
        "param_space": space_lgbm,
        "fixed_kwargs": {"random_state": 42, "verbosity": -1, "n_jobs": -1},
    },
    "xgboost": {
        "model_class": xgb.XGBRegressor,
        "param_space": space_xgb,
        "fixed_kwargs": {"random_state": 42, "n_jobs": -1},
    },
    "linear_regression": {
        "model_class": LinearRegression,
        "param_space": None,
        "fixed_kwargs": {},
        "escalar": True
    },
}

### Función de entrenamiento: clasificación
Para cada mes (1 a 4), `train_model`:
1. Selecciona las variables del mes y aplica el *target encoding* (y el escalado si el modelo es lineal), ajustados con entrenamiento.
2. Busca hiperparámetros con Optuna maximizando la **PR-AUC de la clase reprobado** en validación.
3. Reentrena con los mejores hiperparámetros.
4. Elige el **umbral que maximiza F2** sobre las probabilidades de reprobar en validación.
5. Evalúa en **prueba** con ese umbral y guarda las métricas, la matriz de confusión, la curva precisión-recall, la importancia de variables y el modelo (`.pkl`, que incluye el *encoder*, el *scaler*, el umbral y las columnas, para usarlo en la aplicación).

In [7]:
# Entrenamiento y evaluación de los modelos de clasificación
def train_model(model_key, y, config, out_dir_base ="resultados_clasificacion"):
    """Entrena, ajusta el umbral de decisión y evalúa un modelo de clasificación para cada mes"""
    resultados = []
    importancias = []

    for i in range (1, 5):
        X_train = train[variables[f'mes{i}']]
        X_val = val[variables[f'mes{i}']]
        X_test = test[variables[f'mes{i}']]
        y_train = y_clasificacion_train
        y_val = y_clasificacion_val
        y_test = y_clasificacion_test
        out_dir = f"{out_dir_base}/mes{i}/{model_key}"
        os.makedirs(out_dir, exist_ok=True)

        # Aplicamos encoding
        escalar = config[model_key].get("escalar", False)
        X_train_enc, X_val_enc, te, scaler = preprocess_features(X_train, X_val, y_train, target_type="binary", escalar = escalar)
        X_test_enc = aplicar_encoders(X_test, te, scaler)

        fixed_kwargs = resolve_fixed_kwargs(config[model_key]["fixed_kwargs"], y_train)

        t0 = time.perf_counter() #Empezamos a capturar el tiempo
        # Buscamos hiperparámeteros
        if config[model_key]["param_space"] is not None:
                def objetivo(trial):
                    """Entrena con los parámetros propuestos y devuelve la PR-AUC de reprobado en validación"""
                    params = config[model_key]["param_space"](trial)
                    m = config[model_key]["model_class"](**params, **fixed_kwargs)
                    m.fit(X_train_enc, y_train)
                    idx_neg_trial = list(m.classes_).index(NEG_LABEL)
                    proba_val_neg = m.predict_proba(X_val_enc)[:, idx_neg_trial]
                    return average_precision_score(y_val, proba_val_neg, pos_label=NEG_LABEL)
        
                study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
                study.optimize(objetivo, n_trials=N_TRIALS)
                best_params = study.best_params
                pr_auc_val = study.best_value
                print(f"[{model_key} mes {i}] Mejores hiperparámetros: {best_params}")
                print(f"[{model_key} mes {i}] PR-AUC (validación): {pr_auc_val:.4f}")
        else:
            best_params = {}
            pr_auc_val = None

        tiempo_optuna = time.perf_counter() - t0

        # Entrenamos con los hiperparámetros que encontramos
        t0 = time.perf_counter()
        model = config[model_key]["model_class"](**best_params, **fixed_kwargs)
        model.fit(X_train_enc, y_train)
        tiempo_fit = time.perf_counter() - t0

        print(f"[{model_key} mes {i}] Tiempo Optuna: {tiempo_optuna:.1f}s | "
              f"Tiempo entrenamiento final: {tiempo_fit:.2f}s")
    
        idx_pos = list(model.classes_).index(POS_LABEL)
        idx_neg = list(model.classes_).index(NEG_LABEL)
    
        # Buscamos el mejor umbral
        proba_val_neg = model.predict_proba(X_val_enc)[:, idx_neg]
        umbral, f1_val, prec_val, rec_val = umbral_max_fbeta(y_val, proba_val_neg)
        print(f"[{model_key} mes {i}] Umbral con mejor F2 {umbral:.3f} | "
                f"F2 = {f1_val:.3f} | Precisión = {prec_val:.3f} | Recall = {rec_val:.3f}")

        # Evaluamos en el conjunto de prueba con el umbral
        y_proba_test_pos = model.predict_proba(X_test_enc)[:, idx_pos]
        y_proba_test_neg = model.predict_proba(X_test_enc)[:, idx_neg]
        y_pred_test = np.where(y_proba_test_neg >= umbral, NEG_LABEL, POS_LABEL)
    
        metricas = calcular_metricas_clasificacion(y_test, y_pred_test, y_proba_test_pos, y_proba_test_neg)
        cm = metricas.pop("matriz_confusion")
    
        
        tpr_test = metricas["recall_reprobado"]
        fpr_test = 1 - metricas["recall_aprobado"]     
        prec_test = metricas["precision_reprobado"]
    
        metricas.update({
            "mes": i,
            "modelo": model_key,
            "pr_auc_val": pr_auc_val,
            "umbral_decision": umbral,
            "f1_reprobado_val": f1_val,
            "precision_reprobado_val": prec_val,
            "recall_reprobado_val": rec_val,
            **{f"param_{k}": v for k, v in best_params.items()},
        })
    
        graficar_matriz_confusion(cm, model_key, i, out_dir)
        graficar_pr(y_test, y_proba_test_neg, metricas["pr_auc"], model_key, i, out_dir,
            umbral=umbral, punto_umbral=(tpr_test, prec_test),
            etiqueta_umbral="umbral con mejor F2")
        graficar_importancia(model, X_train_enc.columns, model_key, i, out_dir)

        df_imp = obtener_importancias(model, X_train_enc.columns)
        if df_imp is not None:
            df_imp.to_csv(f"{out_dir}/importancias.csv", index=False)
            df_imp["mes"] = i
            df_imp["modelo"] = model_key
            importancias.append(df_imp)
        metricas["ruta_modelo"] = guardar_modelo(
            model, model_key, i, out_dir,
            te=te, scaler=scaler, umbral=umbral, columnas=X_train.columns
        )
        resultados.append(metricas)

    df_resultados = pd.DataFrame(resultados)
    df_resultados.to_csv(f"{out_dir_base}/resultados_{model_key}.csv", index=False)
    if importancias:
        pd.concat(importancias, ignore_index=True).to_csv(
            f"{out_dir_base}/importancias_{model_key}.csv", index=False
    )


### Función de entrenamiento: regresión
`train_regression_model` sigue la misma lógica, con estas diferencias:
- **Pesos por clase** (`calcular_pesos`): cada observación pesa según si su nota es menor a 61, de modo que reprobados y aprobados aporten lo mismo al error. Sin esto, el modelo se ajustaría casi solo a las notas altas, que son mayoría.
- Optuna minimiza el **RMSE ponderado** en validación. 

In [ ]:
# Entrenamiento y evaluación de los modelos de regresión
def train_regression_model(model_key, y, config, out_dir_base ="resultados_regresion"):
    """Entrena y evalúa un modelo de regresión ponderado para cada mes"""
    resultados = []
    importancias = []
    nombre_etiqueta = y.name if getattr(y, "name", None) else "etiqueta"

    for i in range (1, 5):
        X = datos[variables[f'mes{i}']]
        out_dir = f"{out_dir_base}/mes{i}/{model_key}"
        os.makedirs(out_dir, exist_ok=True)

        escalar = config[model_key].get("escalar", False)
    

        # Dividimos el conjunto
        estratos = (y < 61).astype(int)
        X_train, X_val, X_test, y_train, y_val, y_test = dividir_conjunto(X, y, estratificar=estratos)
        X_train_enc, X_val_enc, te, scaler = preprocess_features(X_train, X_val, y_train, target_type="continuous", escalar = escalar)
        pesos_train = calcular_pesos(y_train)
        pesos_val = calcular_pesos(y_val)
        pesos_test = calcular_pesos(y_test)
        X_test_enc = aplicar_encoders(X_test, te, scaler)

        # Guardamos los conjuntos utilizados
        guardar_conjuntos(out_dir, {
            "train": (X_train, y_train),
            "val":   (X_val,   y_val),
            "test":  (X_test,  y_test),
        }, nombre_etiqueta)
        guardar_conjuntos(out_dir, {
            "train_enc": (X_train_enc, y_train),
            "val_enc":   (X_val_enc,   y_val),
            "test_enc":  (X_test_enc,  y_test),
        }, nombre_etiqueta)

        fixed_kwargs = resolve_fixed_kwargs(config[model_key]["fixed_kwargs"], y_train)

        t0 = time.perf_counter() #Empezamos a capturar el tiempo
        # Buscamos hiperparámeteros
        if config[model_key]["param_space"] is not None:
                def objetivo(trial):
                    """Entrena con los parámetros propuestos y devuelve el RMSE ponderado en validación"""
                    params = config[model_key]["param_space"](trial)
                    m = config[model_key]["model_class"](**params, **fixed_kwargs)
                    m.fit(X_train_enc, y_train, sample_weight=pesos_train)
                    pred_val = m.predict(X_val_enc)
                    return float(np.sqrt(mean_squared_error(y_val, pred_val, sample_weight=pesos_val)))
        
                study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
                study.optimize(objetivo, n_trials=N_TRIALS)
                best_params = study.best_params
                rmse_val_optuna = study.best_value
                print(f"[{model_key} mes {i}] Mejores hiperparámetros: {best_params}")
                print(f"[{model_key} mes {i}] RMSE (validación): {rmse_val_optuna:.4f}")
        else:
            best_params = {}
            rmse_val_optuna = None

        tiempo_optuna = time.perf_counter() - t0

        # Entrenamos con los hiperparámetros que encontramos
        t0 = time.perf_counter()
        model = config[model_key]["model_class"](**best_params, **fixed_kwargs)
        model.fit(X_train_enc, y_train, sample_weight=pesos_train)
        tiempo_fit = time.perf_counter() - t0

        print(f"[{model_key} mes {i}] Tiempo Optuna: {tiempo_optuna:.1f}s | "
              f"Tiempo entrenamiento final: {tiempo_fit:.2f}s")

        pred_val = model.predict(X_val_enc)
        pred_test = model.predict(X_test_enc)

        metricas_val = calcular_metricas_regresion(y_val, pred_val, pesos_val)
        metricas = calcular_metricas_regresion(y_test, pred_test, pesos_test)

        print(f"[{model_key} mes {i}] Prueba → MAE_w = {metricas['mae']:.3f} | "
        f"RMSE_w = {metricas['rmse']:.3f} | "
        f"R² = {metricas['corr']:.3f}")
        
        metricas.update({
            "mes": i,
            "modelo": model_key,
            "rmse_val_optuna": rmse_val_optuna,
            **{f"{k}_val": v for k, v in metricas_val.items()},
            "tiempo_optuna_s": tiempo_optuna,
            "tiempo_fit_s": tiempo_fit,
            **{f"param_{k}": v for k, v in best_params.items()},
        })
    
        # Predicciones del conjunto de prueba
        pd.DataFrame({
            "y_real": np.asarray(y_test),
            "y_pred": pred_test,
            "residuo": np.asarray(y_test) - pred_test,
        }, index=X_test.index).to_csv(f"{out_dir}/predicciones_test.csv")
 
        # Gráficas
        graficar_pred_vs_real(y_test, pred_test, model_key, i, out_dir)
        graficar_residuos(y_test, pred_test, model_key, i, out_dir)
 
        estimador = estimador_final(model)
        graficar_importancia(estimador, X_train_enc.columns, model_key, i, out_dir)
 
        df_imp = obtener_importancias(estimador, X_train_enc.columns)
        if df_imp is not None:
            df_imp.to_csv(f"{out_dir}/importancias.csv", index=False)
            df_imp["mes"] = i
            df_imp["modelo"] = model_key
            importancias.append(df_imp)
 
        metricas["ruta_modelo"] = guardar_modelo(
            model, model_key, i, out_dir,
            te=te, scaler=scaler, umbral=61, columnas=X_train.columns
        )
        resultados.append(metricas)
 
    df_resultados = pd.DataFrame(resultados)
    df_resultados.to_csv(f"{out_dir_base}/resultados_{model_key}.csv", index=False)
    if importancias:
        pd.concat(importancias, ignore_index=True).to_csv(
            f"{out_dir_base}/importancias_{model_key}.csv", index=False
        )

## Clasificación (`Aprobado`)
### Árbol de decisión
Empezamos con un árbol de decisión: es el modelo más simple e interpretable y sirve como punto de referencia.

In [9]:
# Clasificación con árbol de decisión
train_model('decision_tree', y_clasificacion_train, CLASSIFICATION_MODELS)

[decision_tree mes 1] Mejores hiperparámetros: {'criterion': 'entropy', 'max_depth': 17, 'min_samples_split': 42, 'min_samples_leaf': 46, 'max_features': 'sqrt', 'ccp_alpha': 0.0005542686328324747}
[decision_tree mes 1] PR-AUC (validación): 0.8660
[decision_tree mes 1] Tiempo Optuna: 7.2s | Tiempo entrenamiento final: 0.00s
[decision_tree mes 1] Umbral con mejor F2 0.255 | F2 = 0.882 | Precisión = 0.672 | Recall = 0.957
[decision_tree mes 2] Mejores hiperparámetros: {'criterion': 'log_loss', 'max_depth': 16, 'min_samples_split': 58, 'min_samples_leaf': 30, 'max_features': None, 'ccp_alpha': 0.0011719811676632191}
[decision_tree mes 2] PR-AUC (validación): 0.8976
[decision_tree mes 2] Tiempo Optuna: 7.6s | Tiempo entrenamiento final: 0.00s
[decision_tree mes 2] Umbral con mejor F2 0.200 | F2 = 0.888 | Precisión = 0.648 | Recall = 0.979
[decision_tree mes 3] Mejores hiperparámetros: {'criterion': 'log_loss', 'max_depth': 5, 'min_samples_split': 47, 'min_samples_leaf': 15, 'max_features':

### Random Forest
*Ensemble* de árboles entrenados con muestras *bootstrap*; reduce la varianza de un árbol individual.

In [10]:
# Clasificación con Random Forest
train_model('random_forest', y_clasificacion_train, CLASSIFICATION_MODELS)

[random_forest mes 1] Mejores hiperparámetros: {'n_estimators': 289, 'max_depth': 17, 'min_samples_split': 51, 'min_samples_leaf': 3, 'max_features': None, 'criterion': 'log_loss'}
[random_forest mes 1] PR-AUC (validación): 0.9322
[random_forest mes 1] Tiempo Optuna: 488.1s | Tiempo entrenamiento final: 0.36s
[random_forest mes 1] Umbral con mejor F2 0.209 | F2 = 0.889 | Precisión = 0.692 | Recall = 0.957
[random_forest mes 2] Mejores hiperparámetros: {'n_estimators': 627, 'max_depth': 14, 'min_samples_split': 29, 'min_samples_leaf': 1, 'max_features': None, 'criterion': 'entropy'}
[random_forest mes 2] PR-AUC (validación): 0.9252
[random_forest mes 2] Tiempo Optuna: 1001.8s | Tiempo entrenamiento final: 0.85s
[random_forest mes 2] Umbral con mejor F2 0.234 | F2 = 0.896 | Precisión = 0.827 | Recall = 0.915
[random_forest mes 3] Mejores hiperparámetros: {'n_estimators': 101, 'max_depth': 28, 'min_samples_split': 6, 'min_samples_leaf': 6, 'max_features': 'log2', 'criterion': 'entropy'}
[

### LightGBM
*Gradient boosting* con crecimiento de árboles por hoja.

In [11]:
# Clasificación con LightGBM
train_model('lightgbm', y_clasificacion_train, CLASSIFICATION_MODELS)

[lightgbm mes 1] Mejores hiperparámetros: {'n_estimators': 477, 'learning_rate': 0.010240249706482138, 'num_leaves': 56, 'max_depth': 2, 'min_child_samples': 12, 'subsample': 0.6933242619755344, 'colsample_bytree': 0.5160294034835131, 'reg_alpha': 9.9104526079895e-08, 'reg_lambda': 3.4381146780312867e-06}
[lightgbm mes 1] PR-AUC (validación): 0.9312
[lightgbm mes 1] Tiempo Optuna: 170.0s | Tiempo entrenamiento final: 0.09s
[lightgbm mes 1] Umbral con mejor F2 0.131 | F2 = 0.881 | Precisión = 0.630 | Recall = 0.979
[lightgbm mes 2] Mejores hiperparámetros: {'n_estimators': 333, 'learning_rate': 0.10852252042026217, 'num_leaves': 7, 'max_depth': 1, 'min_child_samples': 7, 'subsample': 0.5420241224420308, 'colsample_bytree': 0.6264022963984102, 'reg_alpha': 9.989907653755907e-06, 'reg_lambda': 0.00031541916837120435}
[lightgbm mes 2] PR-AUC (validación): 0.9284
[lightgbm mes 2] Tiempo Optuna: 104.0s | Tiempo entrenamiento final: 0.04s
[lightgbm mes 2] Umbral con mejor F2 0.192 | F2 = 0.89

### XGBoost
*Gradient boosting* con regularización L1/L2.

In [12]:
# Clasificación con XGBoost
train_model('xgboost', y_clasificacion_train, CLASSIFICATION_MODELS)

[xgboost mes 1] Mejores hiperparámetros: {'n_estimators': 188, 'learning_rate': 0.012039831767496189, 'max_depth': 10, 'min_child_weight': 10, 'subsample': 0.9720609630339971, 'colsample_bytree': 0.844683450377908, 'gamma': 1.2006329020335175e-07, 'reg_alpha': 0.004437194748192454, 'reg_lambda': 0.0007548829276028249}
[xgboost mes 1] PR-AUC (validación): 0.9247
[xgboost mes 1] Tiempo Optuna: 184.9s | Tiempo entrenamiento final: 0.10s
[xgboost mes 1] Umbral con mejor F2 0.188 | F2 = 0.876 | Precisión = 0.698 | Recall = 0.936
[xgboost mes 2] Mejores hiperparámetros: {'n_estimators': 385, 'learning_rate': 0.28481434236422387, 'max_depth': 3, 'min_child_weight': 5, 'subsample': 0.598198242402958, 'colsample_bytree': 0.5453404537749249, 'gamma': 3.45518554242572e-08, 'reg_alpha': 4.821905364693244e-07, 'reg_lambda': 1.293779258661829e-08}
[xgboost mes 2] PR-AUC (validación): 0.9505
[xgboost mes 2] Tiempo Optuna: 245.3s | Tiempo entrenamiento final: 0.11s
[xgboost mes 2] Umbral con mejor F2 

### Regresión logística
Modelo lineal de referencia, con penalización L1 o L2 elegida por Optuna.

In [13]:
# Clasificación con regresión logística
train_model('logistic_regression', y_clasificacion_train, CLASSIFICATION_MODELS)

[logistic_regression mes 1] Mejores hiperparámetros: {'C': 0.09072285972637109, 'penalty': 'l1'}
[logistic_regression mes 1] PR-AUC (validación): 0.9143
[logistic_regression mes 1] Tiempo Optuna: 5.3s | Tiempo entrenamiento final: 0.00s
[logistic_regression mes 1] Umbral con mejor F2 0.464 | F2 = 0.858 | Precisión = 0.889 | Recall = 0.851
[logistic_regression mes 2] Mejores hiperparámetros: {'C': 0.007400057463909354, 'penalty': 'l1'}
[logistic_regression mes 2] PR-AUC (validación): 0.9424
[logistic_regression mes 2] Tiempo Optuna: 5.2s | Tiempo entrenamiento final: 0.00s
[logistic_regression mes 2] Umbral con mejor F2 0.258 | F2 = 0.902 | Precisión = 0.687 | Recall = 0.979
[logistic_regression mes 3] Mejores hiperparámetros: {'C': 0.012189713680187515, 'penalty': 'l1'}
[logistic_regression mes 3] PR-AUC (validación): 0.9595
[logistic_regression mes 3] Tiempo Optuna: 5.0s | Tiempo entrenamiento final: 0.00s
[logistic_regression mes 3] Umbral con mejor F2 0.277 | F2 = 0.918 | Precisión 

## Regresión (`NotaFinal`)
### Árbol de decisión

In [14]:
# Regresión con árbol de decisión
train_regression_model('decision_tree', y_regresion_train, REGRESSION_MODELS)


[decision_tree mes 1] Mejores hiperparámetros: {'criterion': 'absolute_error', 'max_depth': 16, 'min_samples_split': 6, 'min_samples_leaf': 22, 'max_features': 'sqrt', 'ccp_alpha': 0.010564700223921425}
[decision_tree mes 1] RMSE (validación): 11.4438
[decision_tree mes 1] Tiempo Optuna: 15.1s | Tiempo entrenamiento final: 0.01s
[decision_tree mes 1] Prueba → MAE_w = 10.509 | RMSE_w = 15.229 | R² = 0.758
[decision_tree mes 2] Mejores hiperparámetros: {'criterion': 'friedman_mse', 'max_depth': 9, 'min_samples_split': 26, 'min_samples_leaf': 9, 'max_features': 'log2', 'ccp_alpha': 0.04971491196903966}
[decision_tree mes 2] RMSE (validación): 8.6100
[decision_tree mes 2] Tiempo Optuna: 7.2s | Tiempo entrenamiento final: 0.00s
[decision_tree mes 2] Prueba → MAE_w = 9.256 | RMSE_w = 13.098 | R² = 0.828
[decision_tree mes 3] Mejores hiperparámetros: {'criterion': 'absolute_error', 'max_depth': 9, 'min_samples_split': 28, 'min_samples_leaf': 4, 'max_features': None, 'ccp_alpha': 0.00622919456

### Random Forest

In [15]:
# Regresión con Random Forest
train_regression_model('random_forest', y_regresion_train, REGRESSION_MODELS)

[random_forest mes 1] Mejores hiperparámetros: {'n_estimators': 297, 'max_depth': 12, 'min_samples_split': 11, 'min_samples_leaf': 1, 'max_features': 'log2', 'criterion': 'absolute_error'}
[random_forest mes 1] RMSE (validación): 11.7380
[random_forest mes 1] Tiempo Optuna: 405.7s | Tiempo entrenamiento final: 0.34s
[random_forest mes 1] Prueba → MAE_w = 9.951 | RMSE_w = 14.543 | R² = 0.782
[random_forest mes 2] Mejores hiperparámetros: {'n_estimators': 66, 'max_depth': 14, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 'log2', 'criterion': 'absolute_error'}
[random_forest mes 2] RMSE (validación): 8.6575
[random_forest mes 2] Tiempo Optuna: 230.0s | Tiempo entrenamiento final: 0.08s
[random_forest mes 2] Prueba → MAE_w = 7.515 | RMSE_w = 9.942 | R² = 0.909
[random_forest mes 3] Mejores hiperparámetros: {'n_estimators': 324, 'max_depth': 17, 'min_samples_split': 10, 'min_samples_leaf': 6, 'max_features': None, 'criterion': 'absolute_error'}
[random_forest mes 3] RMSE (v

### LightGBM

In [16]:
# Regresión con LightGBM
train_regression_model('lightgbm', y_regresion_train, REGRESSION_MODELS)

[lightgbm mes 1] Mejores hiperparámetros: {'n_estimators': 353, 'learning_rate': 0.01818775804704209, 'num_leaves': 7, 'max_depth': 9, 'min_child_samples': 11, 'subsample': 0.905636422595909, 'colsample_bytree': 0.8630481498499656, 'reg_alpha': 0.020604227928958916, 'reg_lambda': 0.0008370171675537779}
[lightgbm mes 1] RMSE (validación): 11.9731
[lightgbm mes 1] Tiempo Optuna: 209.9s | Tiempo entrenamiento final: 0.08s
[lightgbm mes 1] Prueba → MAE_w = 10.714 | RMSE_w = 14.714 | R² = 0.772
[lightgbm mes 2] Mejores hiperparámetros: {'n_estimators': 154, 'learning_rate': 0.01643464997399789, 'num_leaves': 64, 'max_depth': 4, 'min_child_samples': 6, 'subsample': 0.6533260204528623, 'colsample_bytree': 0.7576915085337268, 'reg_alpha': 0.0017657004937694962, 'reg_lambda': 4.04599424515389e-08}
[lightgbm mes 2] RMSE (validación): 8.8872
[lightgbm mes 2] Tiempo Optuna: 166.4s | Tiempo entrenamiento final: 0.14s
[lightgbm mes 2] Prueba → MAE_w = 7.463 | RMSE_w = 9.597 | R² = 0.911
[lightgbm me

### XGBoost

In [17]:
# Regresión con XGBoost
train_regression_model('xgboost', y_regresion_train, REGRESSION_MODELS)

[xgboost mes 1] Mejores hiperparámetros: {'n_estimators': 223, 'learning_rate': 0.04860232914960674, 'max_depth': 2, 'min_child_weight': 4, 'subsample': 0.691544077799416, 'colsample_bytree': 0.9121625550502527, 'gamma': 0.24900751626548376, 'reg_alpha': 0.05683213847351481, 'reg_lambda': 1.3063273790127159e-06}
[xgboost mes 1] RMSE (validación): 11.7226
[xgboost mes 1] Tiempo Optuna: 183.6s | Tiempo entrenamiento final: 0.30s
[xgboost mes 1] Prueba → MAE_w = 10.736 | RMSE_w = 14.477 | R² = 0.778
[xgboost mes 2] Mejores hiperparámetros: {'n_estimators': 263, 'learning_rate': 0.015521415963690641, 'max_depth': 3, 'min_child_weight': 2, 'subsample': 0.8271586062515376, 'colsample_bytree': 0.9648673120617369, 'gamma': 0.8521399853996535, 'reg_alpha': 0.005461728167825493, 'reg_lambda': 1.733171153438945e-05}
[xgboost mes 2] RMSE (validación): 8.5789
[xgboost mes 2] Tiempo Optuna: 251.9s | Tiempo entrenamiento final: 0.30s
[xgboost mes 2] Prueba → MAE_w = 7.210 | RMSE_w = 9.387 | R² = 0.91

### Regresión lineal

In [18]:
# Regresión lineal
train_regression_model('linear_regression', y_regresion_train, REGRESSION_MODELS)

[linear_regression mes 1] Tiempo Optuna: 0.0s | Tiempo entrenamiento final: 0.01s
[linear_regression mes 1] Prueba → MAE_w = 11.545 | RMSE_w = 14.856 | R² = 0.766
[linear_regression mes 2] Tiempo Optuna: 0.0s | Tiempo entrenamiento final: 0.00s
[linear_regression mes 2] Prueba → MAE_w = 6.743 | RMSE_w = 8.958 | R² = 0.922
[linear_regression mes 3] Tiempo Optuna: 0.0s | Tiempo entrenamiento final: 0.00s
[linear_regression mes 3] Prueba → MAE_w = 5.388 | RMSE_w = 6.970 | R² = 0.953
[linear_regression mes 4] Tiempo Optuna: 0.0s | Tiempo entrenamiento final: 0.00s
[linear_regression mes 4] Prueba → MAE_w = 3.015 | RMSE_w = 4.076 | R² = 0.985


## Resultados en el conjunto de prueba (clasificación, clase reprobado)

Fuente: `resultados_clasificacion/resultados_<modelo>.csv`. El umbral es el que maximizó F2 en validación.

| Mes | Modelo | Recall | Precisión | F2 | PR-AUC | Umbral |
|---|---|---|---|---|---|---|
| Enero | LightGBM | 0.896 | 0.683 | 0.843 | 0.849 | 0.131 |
| Enero | Random Forest | 0.896 | 0.705 | 0.850 | 0.842 | 0.209 |
| Enero | Regresión logística | 0.729 | 0.854 | 0.751 | 0.850 | 0.464 |
| Enero | XGBoost | 0.896 | 0.717 | 0.853 | 0.832 | 0.188 |
| Enero | Árbol de decisión | 0.896 | 0.694 | 0.846 | 0.764 | 0.255 |
| Febrero | LightGBM | 0.958 | 0.807 | 0.924 | 0.958 | 0.192 |
| Febrero | Random Forest | 0.917 | 0.815 | 0.894 | 0.951 | 0.234 |
| Febrero | Regresión logística | 0.979 | 0.691 | 0.904 | 0.942 | 0.258 |
| Febrero | XGBoost | 0.979 | 0.758 | 0.925 | 0.951 | 0.136 |
| Febrero | Árbol de decisión | 0.979 | 0.681 | 0.900 | 0.906 | 0.200 |
| Marzo | LightGBM | 0.958 | 0.852 | 0.935 | 0.974 | 0.338 |
| Marzo | Random Forest | 1.000 | 0.814 | 0.956 | 0.978 | 0.222 |
| Marzo | Regresión logística | 1.000 | 0.814 | 0.956 | 0.974 | 0.277 |
| Marzo | XGBoost | 1.000 | 0.727 | 0.930 | 0.984 | 0.137 |
| Marzo | Árbol de decisión | 1.000 | 0.615 | 0.889 | 0.967 | 0.075 |
| Abril | LightGBM | 0.979 | 0.940 | 0.971 | 0.986 | 0.345 |
| Abril | Random Forest | 1.000 | 0.960 | 0.992 | 0.987 | 0.422 |
| Abril | Regresión logística | 0.979 | 0.940 | 0.971 | 0.985 | 0.392 |
| Abril | XGBoost | 0.979 | 0.959 | 0.975 | 0.981 | 0.303 |
| Abril | Árbol de decisión | 0.979 | 0.940 | 0.971 | 0.993 | 0.800 |

**Observaciones:**
- El desempeño mejora conforme avanza el semestre: con la información de enero el mejor F2 es ≈ 0.85 y en abril todos los modelos superan 0.97.
- Desde febrero (con el primer parcial), todos los modelos detectan más del 90 % de los reprobados.
- Las diferencias entre algoritmos son pequeñas y ninguno es el mejor en todos los meses.
- La aplicación usa los modelos **Random Forest**: los archivos de `app/models/` son los mismos que `resultados_clasificacion/mesN/random_forest/`.